# 第2回・発展宿題：講座用語集のLoRA学習・検索との比較

公式演習とは別の発展用補助ノートブックです。**T4 GPU**で実行してください。あらかじめFork側で `uv run python scripts/lec02_glossary_prepare.py` を実行し、生成した `glossary_sft.jsonl` と `glossary_eval.json` を用意します。

実験の目的は、同じ12問に対して **学習前・LoRA学習後・単純辞書検索** を比較することです。未知用語は「用語集にないと判断できるか」を評価します。

GPUでの動作・スコアはまだ実測していません。結果を推定して記入しないでください。

## 0. パッケージを講座で確認された版に固定する

In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
PINS = {"transformers":"5.18.0", "peft":"0.21.1", "trl":"1.14.2",
        "accelerate":"1.15.0", "datasets":"5.1.0"}
spec = " ".join(f'"{k}=={v}"' for k,v in PINS.items())
%pip uninstall -y -q torchao
%pip install -q {spec}

## 1. データのアップロード・学習条件の固定

In [ ]:
from google.colab import files
from pathlib import Path
import json, time, random
import torch, pandas as pd
from IPython.display import display
assert torch.cuda.is_available(), "ランタイム → T4 GPU を選択してください"

files.upload()  # glossary_sft.jsonl と glossary_eval.json の2つを選択
training = [json.loads(line) for line in Path("glossary_sft.jsonl").read_text(encoding="utf-8").splitlines() if line.strip()]
cases = json.loads(Path("glossary_eval.json").read_text(encoding="utf-8"))
assert training and cases
CONFIG = {"base_model":"Qwen/Qwen3.5-0.8B",
          "revision":"2fc06364715b967f1860aea9cf38778875588b17",
          "seed":42, "num_train_epochs":4, "lr":2e-4,
          "lora_r":16, "lora_alpha":32, "max_new_tokens":200, "do_sample":False}
WORK = Path("/content/lec02_glossary"); WORK.mkdir(parents=True, exist_ok=True)
random.seed(CONFIG["seed"]); torch.manual_seed(CONFIG["seed"])
print("学習用:", len(training), "評価用:", len(cases), CONFIG)

## 2. 学習前のモデルで12問を評価する

この発展課題の教師データは自然文Q&Aです。公式演習の `answer` / `needs_review` JSON評価とは分けます。生成条件は学習前後で同じにします。

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
tok = AutoTokenizer.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"])
if tok.pad_token is None: tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], dtype=torch.float32).to("cuda")
model.eval()

def answer(model, question):
    # 学習データと同じ user メッセージのみをモデルに渡す
    prompt = tok.apply_chat_template([{"role":"user", "content":question}], tokenize=False,
                                     add_generation_prompt=True, enable_thinking=False)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=CONFIG["max_new_tokens"],
                             do_sample=CONFIG["do_sample"], pad_token_id=tok.pad_token_id)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def evaluate(model, label):
    t0 = time.perf_counter()
    rows = [{**item, "raw":answer(model, item["question"])} for item in cases]
    payload = {"label":label, "seconds":round(time.perf_counter()-t0,2), "rows":rows}
    (WORK/f"eval_{label}.json").write_text(json.dumps(payload,ensure_ascii=False,indent=2),encoding="utf-8")
    print(label, "評価秒数:",payload["seconds"])
    display(pd.DataFrame([{"種別":r["category"],"質問":r["question"],"モデル回答":r["raw"]} for r in rows]))
    return payload

before = evaluate(model, "before")

## 3. LoRAで学習する（116例を4エポックから開始）

In [ ]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import Dataset
import transformers, peft, trl
model.config.use_cache = False
lora = LoraConfig(r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"], lora_dropout=0.05,
                  bias="none", task_type="CAUSAL_LM",
                  target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"])
params = SFTConfig(output_dir=str(WORK/"ckpt"),
                   num_train_epochs=CONFIG["num_train_epochs"],
                   per_device_train_batch_size=4, gradient_accumulation_steps=2,
                   learning_rate=CONFIG["lr"], lr_scheduler_type="cosine",
                   warmup_steps=6, logging_steps=5, save_steps=20, save_total_limit=2,
                   max_length=512, packing=False, fp16=False, report_to="none", seed=CONFIG["seed"])
trainer = SFTTrainer(model=model, args=params,
                     train_dataset=Dataset.from_list(training),
                     peft_config=lora, processing_class=tok)
trainer.model.print_trainable_parameters()
t0 = time.perf_counter()
trainer.train()
train_seconds = round(time.perf_counter()-t0, 2)
losses = [row["loss"] for row in trainer.state.log_history if "loss" in row]
final_loss = losses[-1] if losses else None
tuned = trainer.model
tuned.config.use_cache = True
tuned.eval()
tuned.save_pretrained(str(WORK/"adapter"))
print("学習秒数:",train_seconds,"ステップ:",trainer.state.global_step,"最終記録loss:",final_loss)

## 4. 同じ12問で学習後を評価し、保存する

`manual_review.json` を見て正答を**人手で採点**してください。モデルが一般知識で未知用語を説明しただけでは、講座用語集に根拠があるとは言えません。検索ベースラインの結果は `lexical_baseline.json` を参照します。

In [ ]:
after = evaluate(tuned, "after")
review = []
for b,a in zip(before["rows"], after["rows"]):
    assert b["id"] == a["id"], "評価データの順序が違います"
    review.append({"id":b["id"], "category":b["category"], "question":b["question"],
                   "expected_term_id":b["expected_term_id"], "gold_definition":b["gold_definition"],
                   "before_answer":b["raw"], "after_answer":a["raw"],
                   "before_correct":None, "after_correct":None, "memo":""})
(WORK/"manual_review.json").write_text(json.dumps(review,ensure_ascii=False,indent=2),encoding="utf-8")
result = {"executed_training_myself":True, "method":"LoRA（FP32基盤モデル）",
          "config":CONFIG, "training_rows":len(training), "evaluation_rows":len(cases),
          "train_seconds":train_seconds, "training_steps":trainer.state.global_step,
          "final_logged_loss":final_loss,
          "before_eval_seconds":before["seconds"], "after_eval_seconds":after["seconds"],
          "versions":{"torch":torch.__version__,"transformers":transformers.__version__,
                      "peft":peft.__version__,"trl":trl.__version__},
          "correct_rate":"未採点（manual_review.json を照合して算出する）"}
(WORK/"advanced_result.json").write_text(json.dumps(result,ensure_ascii=False,indent=2),encoding="utf-8")
display(pd.DataFrame([{"種別":x["category"],"質問":x["question"],
                       "学習前":x["before_answer"],"学習後":x["after_answer"],
                       "正解定義":x["gold_definition"]} for x in review]))
print("実測結果は保存済み。正答率は未採点です。")

## 5. 結果をダウンロードする

取得後は必ず **ランタイム → ランタイムを接続解除して削除** を実行してください。

In [ ]:
from google.colab import files
for filename in ["advanced_result.json","manual_review.json","eval_before.json","eval_after.json"]:
    files.download(str(WORK/filename))